In [ ]:
from pathlib import Path
import runpy

def _find_notebook_bootstrap(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "notebooks" / "_bootstrap.py"
        if direct.exists():
            return direct
        nested = candidate / "abstractgraph-generative" / "notebooks" / "_bootstrap.py"
        if nested.exists():
            return nested
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_find_notebook_bootstrap(Path.cwd())))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


# Story Graph Closed Alphabet Induction

This notebook induces bounded vocabularies for `entities`, `relations`, `intentions`, and `events` from a corpus of stories:

1. First pass: LLM extracts candidate terms per story.
2. Frequency tracking: aggregate corpus counts.
3. Second pass: LLM consolidates/clusters terms into closed alphabets with user-defined sizes.


In [1]:
import json
import os
from pathlib import Path

from abstractgraph_generative.story import (
    ask_llm,
    build_raw_term_inventory,
    consolidate_category_terms,
    dump_json,
    induce_closed_alphabets,
    load_aesop_fables_from_gutenberg,
)

MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")

DICTIONARY_DIR = "generative/story_graph/resources"


In [2]:
# User-defined target alphabet sizes
N_ENTITIES = 100
N_RELATIONS = 40
N_GOALS = 40
N_INTENTIONS = 80
N_EVENTS = 160
MIN_FREQ = 3

# Corpus size for induction run
N_STORIES = 300


In [3]:
fables = load_aesop_fables_from_gutenberg()
stories = fables[:N_STORIES]
print(f"Loaded {len(fables)} stories total; using {len(stories)} for induction.")


Loaded 313 stories total; using 300 for induction.


In [4]:
raw_counts = build_raw_term_inventory(
    stories,
    ask_llm_fn=ask_llm,
    model=MODEL,
)

for key in ["entities", "relations", "goals", "intentions", "events"]:
    print(f"\nTop raw {key}:")
    for term, freq in raw_counts.get(key, {}).most_common(20):
        print(f"  {term}: {freq}")



Top raw entities:
  Lion: 38
  Fox: 37
  Man: 34
  Wolf: 28
  Ass: 24
  Dog: 21
  Sheep: 16
  Shepherd: 16
  Tree: 15
  Eagle: 13
  Bird: 12
  Bull: 12
  Beast: 10
  Hare: 10
  Traveler: 10
  Goat: 10
  Horse: 10
  Owner: 9
  Crow: 9
  Mother: 9

Top raw relations:
  ENEMY_OF: 52
  FRIEND_OF: 35
  OWNER_OF: 23
  PARENT_OF: 17
  COMPANION_OF: 12
  PREY_OF: 9
  MASTER_OF: 7
  REQUESTS: 6
  CARES_FOR: 6
  ALLIED_WITH: 5
  CAPTURED_BY: 5
  RIVAL_OF: 4
  NEIGHBOR_OF: 4
  HELPED_BY: 4
  HEALED_BY: 4
  GUARDIAN_OF: 4
  DISPUTE: 3
  DISPUTE_OVER: 3
  CAUGHT_IN: 3
  HAS: 3

Top raw goals:
  SURVIVE: 120
  ESCAPE: 10
  AVENGE: 10
  PROTECT: 6
  OBTAIN_RESOURCE: 5
  UNDERSTAND: 4
  EAT: 4
  DECEIVE: 4
  CATCH_FISH: 3
  OBTAIN_HELP: 3
  AVOID_PUNISHMENT: 3
  HEAL: 3
  PROLONG_LIFE: 3
  DEFEAT_ENEMY: 3
  ENTERTAIN: 3
  OBTAIN_FOOD: 3
  AVOID_HARM: 3
  SATISFY_HUNGER: 3
  SPARE_LIFE: 2
  PROVE_SUPERIORITY: 2

Top raw intentions:
  PROTECT_SELF: 60
  AVOID_DANGER: 6
  SPARE_LIFE: 6
  BOAST: 4
  SAVE

In [5]:
induced = induce_closed_alphabets(
    stories,
    ask_llm_fn=ask_llm,
    model=MODEL,
    n_entities=N_ENTITIES,
    n_relations=N_RELATIONS,
    n_goals=N_GOALS,
    n_intentions=N_INTENTIONS,
    n_events=N_EVENTS,
    min_freq=MIN_FREQ,
    dictionary_dir=DICTIONARY_DIR,
    save_filename="induced_closed_alphabet.json",
)

print("Targets:")
print(dump_json(induced["targets"]))


Targets:
{
  "entities": 100,
  "relations": 40,
  "goals": 40,
  "intentions": 80,
  "events": 160,
  "min_freq": 3
}


In [6]:
for key in ["entities", "relations", "goals", "intentions", "events"]:
    block = induced["consolidated"][key]
    canonical = block["canonical_terms"]
    counts = block["canonical_counts"]
    print(f"\nCanonical {key} ({len(canonical)} terms):")
    for term in canonical:
        print(f"  {term}: {counts.get(term, 0)}")



Canonical entities (1 terms):
  Lion: 739

Canonical relations (28 terms):
  ADVISES: 3
  ALLIED_WITH: 5
  APPROACHES: 3
  CAPTURED_BY: 5
  CARES_FOR: 3
  CARRIES: 3
  CATCHES: 3
  CAUGHT_BY: 6
  COMPANION_OF: 12
  DISPUTE: 6
  ENEMY_OF: 61
  ENVY_OF: 4
  FRIEND_OF: 33
  FRIGHTENED_BY: 3
  HEALED_BY: 3
  HEARD: 3
  KILLED_BY: 3
  MASTER_OF: 8
  NEIGHBOR_OF: 5
  OWNER_OF: 24
  PARENT_OF: 18
  PARTNER_OF: 3
  PART_OF: 3
  PREY_OF: 11
  REQUESTS: 6
  RIVAL_OF: 6
  SERVANT_OF: 3
  SIBLING_OF: 4

Canonical goals (17 terms):
  AVENGE: 14
  AVOID_HARM: 3
  AVOID_PUNISHMENT: 3
  CATCH_FISH: 3
  DEFEAT_ENEMY: 4
  EAT: 3
  ENTERTAIN: 3
  ESCAPE: 11
  HEAL: 4
  OBTAIN_FOOD: 4
  OBTAIN_RESOURCE: 6
  PROLONG_LIFE: 3
  PROTECT: 6
  PROTECT_FLOCK: 3
  SATISFY_HUNGER: 3
  SURVIVE: 121
  UNDERSTAND: 3

Canonical intentions (10 terms):
  ASSIST_OTHER: 3
  AVOID_DANGER: 4
  BOAST: 3
  DECEIVE_OTHERS: 7
  DEFEND_SELF: 4
  HELP_OTHER: 4
  PROTECT_SELF: 58
  SAVE_SELF: 3
  SPARE_LIFE: 5
  WARN_OTHER: 7

Ca

In [7]:
out_path = Path(DICTIONARY_DIR) / "induced_closed_alphabet.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(induced, indent=2), encoding="utf-8")
print(f"Saved induction result: {out_path}")


Saved induction result: generative/story_graph/resources/induced_closed_alphabet.json
